# TRUEGUARD Capstone Implementation
## Multi-Signal Hallucination Detection & Mitigation Framework

This notebook implements the exact methodology proposed in the TRUEGUARD review paper.
We use a lightweight open-source LLM (TinyLlama) to demonstrate real-time extraction of internal states, attention maps, and entropy, which are fused to detect hallucinations.

In [ ]:
!pip install -q transformers torch accelerate scipy numpy

In [ ]:
import torch
import numpy as np
from scipy.stats import entropy
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load Model with Hidden States & Attentions Enabled
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" # Lightweight model for fast Colab execution
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",               # Automatically use T4 GPU if available
    torch_dtype=torch.float16,       # Half-precision for memory efficiency
    output_hidden_states=True,       # CRITICAL: For Gap 1 (Internal State tracking)
    output_attentions=True           # CRITICAL: For Gap 1 (Attention anomaly detection)
)
model.eval()
print("TRUEGUARD Engine: Model Loaded Successfully.")

In [ ]:
def calculate_entropy(logits):
    """Calculates Semantic Energy / Entropy (Solves the Entropy Gap)."""
    probs = torch.nn.functional.softmax(logits, dim=-1)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    entropy_val = -torch.sum(probs * log_probs, dim=-1)
    return entropy_val.mean().item()

def calculate_attention_anomaly(attentions):
    """Measures attention focus on input context (Solves the Intrinsic Hallucination Gap)."""
    last_layer_attn = attentions[-1][0].mean(dim=0) # Shape: [seq_len, seq_len]
    concentration = last_layer_attn.mean().item()
    return 1.0 - concentration

def calculate_distribution_shift(hidden_states):
    """Measures drift in hidden representations (Solves the Temporal Drift Gap)."""
    first_layer_hs = hidden_states[0][0][-1].float().detach().cpu().numpy()
    last_layer_hs = hidden_states[-1][0][-1].float().detach().cpu().numpy()
    shift = 1.0 - np.dot(first_layer_hs, last_layer_hs) / (np.linalg.norm(first_layer_hs) * np.linalg.norm(last_layer_hs))
    return float(shift)

In [ ]:
def trueguard_generate_and_detect(prompt_text, threshold=0.15):
    """
    The Core TRUEGUARD Pipeline (Single Forward Pass)
    Generates text while actively extracting signals and computing risk.
    """
    print(f"\n[USER PROMPT]: {prompt_text}")
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            return_dict_in_generate=True,
            output_scores=True,          # For Logits/Entropy
            output_hidden_states=True,   # For Distribution Shift
            output_attentions=True       # For Attention Anomaly
        )
        
    generated_ids = outputs.sequences[0][inputs.input_ids.shape[-1]:]
    response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # --- EXTRACTION (Single Pass) ---
    logits = outputs.scores[-1]
    entropy_signal = calculate_entropy(logits)
    
    hidden_states = outputs.hidden_states[-1]
    shift_signal = calculate_distribution_shift(hidden_states)
    
    attentions = outputs.attentions[-1]
    attention_signal = calculate_attention_anomaly(attentions)
    
    # --- FUSION LAYER (Mathematical Weighting) ---
    w_entropy = 0.4
    w_shift = 0.35
    w_attn = 0.25
    
    norm_entropy = min(entropy_signal / 10.0, 1.0)
    
    final_risk_score = (w_entropy * norm_entropy) + (w_shift * shift_signal) + (w_attn * attention_signal)
    
    print("\n[TRUEGUARD ANALYSIS]:")
    print(f"- Entropy Score (Uncertainty): {norm_entropy:.4f}")
    print(f"- Distribution Shift Score:    {shift_signal:.4f}")
    print(f"- Attention Anomaly Score:     {attention_signal:.4f}")
    print(f">>> FINAL HALLUCINATION RISK:  {final_risk_score:.4f} / 1.0")
    
    # --- MITIGATION PIPELINE ---
    if final_risk_score > threshold:
        print("\n[⚡ MITIGATION TRIGGERED]: Risk exceeded safety threshold!")
        print("[TRUEGUARD INTERCEPTION]: Response withheld due to high probability of hallucination.")
        print("\n> Final Safe Output: 'I don't have enough confident information to answer this reliably. Please consult a verified source.'")
    else:
        print("\n[✅ TRUEGUARD CLEARED]: Generation is stable and confident.")
        print(f"\n> Final Output: '{response_text.strip()}'")
        
    return final_risk_score


### Test Case 1: Simple Factual Query (Low Hallucination Risk)

In [ ]:
prompt1 = "<|system|>\nYou are a helpful assistant.\n<|user|>\nWhat is the capital of France?\n<|assistant|>\n"
trueguard_generate_and_detect(prompt1, threshold=0.15)

### Test Case 2: Tricky/Hallucination-Inducing Query (High Hallucination Risk)

In [ ]:
prompt2 = "<|system|>\nYou are a helpful assistant.\n<|user|>\nWho was the first person to walk on Mars in 1965?\n<|assistant|>\n"
trueguard_generate_and_detect(prompt2, threshold=0.15)